# Atividade 4 -- Tópicos para Computação 1 -- 2026.1

- Escola Superior de Tecnologia
- Profa. Dra. Elloá B. Guedes (ebgcosta@uea.edu.br)
- www.elloaguedes.com
- github.com/elloa
- Data: 19 de março de 2026

## Descrição
A atividade consiste em construir uma rede neural convolucional usando transfer learning da rede Inception V3 para classificar o Stanford Dogs Dataset, um dataset com imagens de 120 raças de cachorro de todo o mundo e mais de 20 mil imagens para treino e teste

### Requisitos:

- Uso de Transfer Learning
- Uso de Taxa de Aprendizado Adaptativa
- Data Augmentation
- L2 regularizatiom
- Early Stopping
- Gráficos de loss/acurácia no treino e validação
- Métricas no conjunto de teste
- Elaboração de um conjunto de slides com os resultados dos experimentos e apresentação da arquitetura
- Monitoramento do tempo de treinamento
- Quantidade de parâmetros totais e ajustáveis

### Equipe:

- Ana Flavia de Castro Segadillha da Silva
- Davi Aguiar Moreira
- Guilherme Goncalves Moraes
- Luiz Fernando Borges Brito
- Pedro Vitor Barros Maranhao

## Bibliotecas

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from torchvision.models import inception_v3, Inception_V3_Weights
import random
import shutil
from torch.amp import autocast,GradScaler

In [3]:
print(torch.cuda.is_available())

True


## Extração e filtragem do dataset
- O stanford dogs dataset é uma tarefa de classificação multiclasse que contém fotos de 120 diferentes raças de cães.
- A documentação oficial pode ser obtida aqui: http://vision.stanford.edu/aditya86/ImageNetDogs/
- A partição oficial está no servidor da disciplina (/opt/topicos/data).

In [16]:
base_origem = '/opt/topicos/data'
base_destino = os.path.expanduser('~/data')  #voltar para home do usuário

for pasta in ['train', 'val', 'test']:
    origem = os.path.join(base_origem, pasta)
    destino = os.path.join(base_destino, pasta)

    if not os.path.exists(destino):
        shutil.copytree(origem, destino)

train_path = os.path.join(base_destino, 'train')
test_path  = os.path.join(base_destino, 'test')
valid_path = os.path.join(base_destino, 'val')


In [ ]:
#Função para retirar "n[id]" do início das pastas buscando por "-"
def tirar_id(path):
    for item in os.listdir(path):
        nome_bruto = os.path.join(path, item)
        segmentado = item.split("-", 1)     
        novo_nome = segmentado[1]
        nome_limpo = os.path.join(path, novo_nome)   
        os.rename(nome_bruto, nome_limpo)

tirar_id(test_path)
tirar_id(valid_path)
tirar_id(train_path)

## Transformações com Data Augmentation

- As imagens precisam ser padronizadas, então usaremos as dimensões 299×299 exigidas pelo `Inception V3` e a normalização das cores;
- Data augmentation é uma técnica aplicada somente na partição de teste, que aplica variações nas imagens de forma aleatória com a intenção de aumentar a quantidade de exemplos de cada classe e evitar que o modelo memorize exemplos específicos: https://docs.pytorch.org/vision/0.22/transforms.html;
- Para cada partição, se cria um ImageFolder:
    - https://docs.pytorch.org/vision/main/generated/torchvision.datasets.ImageFolder.html#torchvision.datasets.ImageFolder;
- Para `train`, usa-se um Data Loader com batch de 32 e com randomização;
- Para `val`, usa-se um Data Loader com batch de 32 e sem randomização;
- Para `test`, usa-se um Data Loader com batch de 1 e sem randomização;
    - https://docs.pytorch.org/docs/stable/data.html.

In [17]:
train_transform = transforms.Compose([transforms.Resize((320, 320)),transforms.RandomCrop(299), transforms.RandomHorizontalFlip(),
                                      transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), ## varia brilho, contraste, saturação e matiz
                                      transforms.RandomRotation(10), transforms.ToTensor(),
                                      transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])])
## obs: o conjunto de teste não recebe o data augmentation, só a normalização.

test_transform = transforms.Compose([transforms.Resize((299, 299)), transforms.ToTensor(),
                                     transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])])

train_dataset = ImageFolder(root=train_path, transform=train_transform)
val_dataset = ImageFolder(root=valid_path, transform=test_transform)
test_dataset  = ImageFolder(root=test_path,  transform=test_transform)

train_load = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_load = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_load  = DataLoader(test_dataset,  batch_size=1,  shuffle=False)

images, labels = next(iter(train_load))
print("Batch shape:", images.shape, labels.shape)

Batch shape: torch.Size([32, 3, 299, 299]) torch.Size([32])


## Utilizando o modelo Inception V3

- O modelo utilizado pela equipe é o `Inception V3`. Sua primeira versão, nomeada como `GoogLeNet`, foi a vencedora do ImageNet 2014;
- A sua arquitetura é, seguindo esta ordem:
    - Para sua _stem_
        - Conv 1: 32 filtros de 3×3, com stride 2, sem padding;
        - Conv 2: 32 filtros de 3×3, com stride 1, sem padding;
        - Conv 3: 64 filtros de 3×3, com stride 1, com padding;
        - Max Pool 1: Redução espacial com janelas 3×3 com stride 2;
        - Conv 4: 80 filtros de 1×1, com stride 1, sem padding;
        - Conv 5: 192 filtros de 3×3, com stride 1, sem padding;
        - Max Pool 2: Outra redução de 3×3 com stride 2.
    - Para bloco `Inception A`, repetido 3 vezes e com os ramos paralelos:
        - Ramo 1: Convolução 1×1;
        - Ramo 2: Convolução 1×1, seguida de convolução 3×3;
        - Ramo 3: Convolução 1×1, seguida de convolução 3×3, seguida de outra convolução 3×3;
        - Ramo 4: Avg Pool de 3×3, seguida de convolução 1×1.
    - Para bloco `Reduction A`, com os ramos paralelos:
        - Ramo 1: Convolução 3×3;
        - Ramo 2: Max Pool 3×3;
        - Ramo 3: Convolução 1×1, seguida de convolução 3×3, seguida de outra convolução 3×3.
    - Para bloco `Inception B`, repetido 4 vezes e tem os ramos paralelos:
        - Ramo 1: Convolução 1×1, seguida de convolução 1×7, seguida de convolução 7×1;
        - Ramo 2: Convolução 1×1, seguida duas vezes a convolução 1×7 seguida de convolução 7×1;
        - Ramo 3: Convolução 1×1;
        - Ramo 4: Avg Pool de 3×3, seguida de convolução 1×1.
    - Para bloco auxiliar, usado apenas durante o treinamento:
        - Avg Pool: Janela de 5×5 com stride 3;
        - Conv 1: 128 filtros de 1×1 com stride 1;
        - Flatten: Achatamento do tensor para 758 elementos;
        - Dense 1: Camada densa de 758 elementos com ativação ReLu, aplicado dropout de 0.2;
        - Dense 2: Camada densa de 1000 elementos, aplicado softmax;
    - Para bloco `Reduction B`, com os ramos paralelos:
        - Ramo 1: Convolução 1×1, seguida de convolução 3×3;
        - Ramo 2: Max Pool 3×3;
        - Ramo 3: Convolução 1×1, seguida de convolução 1×7, seguida de convolução 7×1, seguida de convolução 3×3.
    - Para bloco `Inception C`, repetido 2 vezes e tem os ramos paralelos:
        - Ramo 1: Convolução 1×1;
        - Ramo 2: Convolução 1×1, seguida de convolução 3×3, seguida de uma concatenação entre uma convolução 1×3 e uma convolução 3×1;
        - Ramo 3: Max Pool 3×3, seguida de convolução 1×1;
        - Ramo 4: Convolução 1×13, seguida de uma concatenação entre uma convolução 1×3 e uma convolução 3×1.
- Para este trabalho, foi usado o modelo pré-treinado e substituindo apenas as camadas de saída para as 120 classes.
    -  Transfer Learning: https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html

In [7]:
model = inception_v3(weights=Inception_V3_Weights.DEFAULT, aux_logits=True)
classes = 120
#Camada de saída principal e auxiliar
model.fc = nn.Linear(model.fc.in_features, classes)
model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, classes)

In [11]:
for param in model.parameters():
    param.requires_grad = False

# liberar só as camadas de saída para treino
for param in model.fc.parameters():
    param.requires_grad = True
for param in model.AuxLogits.parameters():
    param.requires_grad = True

<h2>Configurações do treino</h2>

In [12]:
epocas = 120
aprendizado = 1e-3
momentum = 0.9
paciencia= 10

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=aprendizado, momentum=momentum,weight_decay=1e-4) #Adição do weight_decay para L2 regularization

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epocas, eta_min=1e-6) ## taxa de aprendizado adaptativa

csv_file_path = "training_metrics.csv"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model  = model.to(device)

pd.DataFrame(columns=["Epoch", "Train_Loss", "Train_Accuracy", "Val_Loss", "Val_Accuracy", "LR"]).to_csv(csv_file_path, index=False)

cuda


In [24]:
scaler = torch.amp.GradScaler("cuda")
last_val_acc= 0.0
for epoch in range(epocas):
    if paciencia == 0:
        break
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in train_load:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()

        with torch.autocast(device_type="cuda"):
            outputs = model(inputs)
            loss1 = criterion(outputs.logits, targets)
            loss2 = criterion(outputs.aux_logits, targets)
            loss = loss1 + 0.3 * loss2
            
        scaler.scale(loss).backward() 
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.logits.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, targets in val_load:
            inputs, targets = inputs.to(device), targets.to(device)

            with torch.autocast(device_type="cuda"):
                outputs = model(inputs)
                logits=outputs
                loss = criterion(logits, targets)

            val_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            val_total += targets.size(0)
            val_correct += predicted.eq(targets).sum().item()

    val_loss /= len(val_dataset)
    val_accuracy = 100. * val_correct / val_total
    current_val_acc = val_accuracy
    if last_val_acc >= current_val_acc:   #Early stopping
        paciencia= paciencia-1
    last_val_acc = current_val_acc

    scheduler.step()  ## atualizar lr ao fim de cada época

    train_loss = running_loss / len(train_dataset)
    train_accuracy = 100. * correct / total
    current_lr = scheduler.get_last_lr()[0]

    pd.DataFrame([[epoch + 1, train_loss, train_accuracy, val_loss, val_accuracy, current_lr]],
                 columns=["Epoch", "Train_Loss", "Train_Accuracy","Val_Loss", "Val_Accuracy", "LR"]).to_csv(csv_file_path, mode='a', header=False, index=False)

    print(f"Epoch {epoch+1:3d}/{epocas} | Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.2f}% |  Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.2f}% | "f"LR: {current_lr:.2e}")
    
torch.save(model.state_dict(), "Modelo_Inception.pth")

Epoch   1/120 | Train Loss: 2.1914 | Train Acc: 68.24% |  Val Loss: 1.4032 | Val Acc: 76.58% | LR: 9.99e-04
Epoch   2/120 | Train Loss: 2.0314 | Train Acc: 68.99% |  Val Loss: 1.3517 | Val Acc: 77.58% | LR: 9.98e-04
Epoch   3/120 | Train Loss: 1.9011 | Train Acc: 69.68% |  Val Loss: 1.1989 | Val Acc: 78.17% | LR: 9.97e-04
Epoch   4/120 | Train Loss: 1.7997 | Train Acc: 70.16% |  Val Loss: 1.1617 | Val Acc: 79.08% | LR: 9.96e-04
Epoch   5/120 | Train Loss: 1.7365 | Train Acc: 70.46% |  Val Loss: 1.1307 | Val Acc: 78.33% | LR: 9.94e-04
Epoch   6/120 | Train Loss: 1.6510 | Train Acc: 71.34% |  Val Loss: 1.1041 | Val Acc: 78.08% | LR: 9.92e-04
Epoch   7/120 | Train Loss: 1.5927 | Train Acc: 71.62% |  Val Loss: 1.0341 | Val Acc: 78.92% | LR: 9.89e-04
Epoch   8/120 | Train Loss: 1.5555 | Train Acc: 71.31% |  Val Loss: 0.9902 | Val Acc: 79.67% | LR: 9.86e-04
Epoch   9/120 | Train Loss: 1.5160 | Train Acc: 71.94% |  Val Loss: 0.9470 | Val Acc: 79.50% | LR: 9.83e-04
Epoch  10/120 | Train Loss: 

<h2>Métricas do treino</h2>

In [13]:
model.load_state_dict(torch.load("Modelo_Inception.pth", map_location=device))
model = model.to(device)
model.eval()

Inception3(
  (Conv2d_1a_3x3): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2a_3x3): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2b_3x3): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_4a_3x3): BasicConv2d(
    (conv): Conv2d(80, 192, kernel_size=(3, 3), stri

In [19]:
pred = []
label = []

model.eval()
with torch.no_grad():
    for inputs, labels in test_load:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        pred.extend(predicted.cpu().numpy())
        label.extend(labels.cpu().numpy())

print(classification_report(label, pred, target_names=test_dataset.classes))

                                precision    recall  f1-score   support

                  Afghan_hound       0.93      0.96      0.94       139
           African_hunting_dog       0.92      0.94      0.93        69
                      Airedale       0.90      0.71      0.79       102
American_Staffordshire_terrier       0.67      0.48      0.56        64
                   Appenzeller       0.62      0.55      0.58        51
            Australian_terrier       0.76      0.67      0.71        96
            Bedlington_terrier       0.87      0.88      0.87        82
          Bernese_mountain_dog       0.94      0.89      0.91       118
              Blenheim_spaniel       0.95      0.93      0.94        88
                 Border_collie       0.76      0.84      0.80        50
                Border_terrier       0.97      0.86      0.91        72
                   Boston_bull       0.93      0.78      0.85        82
          Bouvier_des_Flandres       0.80      0.70      0.74  

<h2>Avaliação do Modelo</h2>

<h2>Conclusões</h2>